In [2]:
#Consecutive login days
import sqlite3

conn = sqlite3.connect("Employee_login.db")
cursor = conn.cursor()

print("Connection established and database created successfully")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Employees(
    Emp_id INT,
    Login_date DATE)
""")

conn.commit()
print("Table created successfully")

employee_data = [(101, '2026-09-01'),
(101, '2026-09-02'),
(101, '2026-09-03'),
(102, '2026-09-01'),
(102, '2026-09-03'),
(102, '2026-09-05'),
(103, '2026-09-04'),
(103, '2026-09-05'),
(103, '2026-09-06'),
(104, '2026-09-02'),
(104, '2026-09-03'),
(104, '2026-09-05'),
(105, '2026-09-07'),
(105, '2026-09-08'),
(105, '2026-09-09'),
(106, '2026-09-01'),
(106, '2026-09-02'),
(106, '2026-09-04'),
(107, '2026-09-10'),
(107, '2026-09-11')]

cursor.execute("SELECT COUNT(*) FROM Employees")
count = cursor.fetchone()[0]

if count == 0:
    cursor.executemany("""
        INSERT INTO Employees (Emp_id, Login_date)
        VALUES (?,?)""", employee_data)
    conn.commit()
    print("Data entered successfully")
else:
    print("Data already exists")


cursor.execute("""
SELECT Emp_id
FROM(
SELECT *,julianday(login_date) - julianday(lag_date) AS first,
        julianday(lead_date) - julianday(login_date) AS second
FROM(
    SELECT *,
        LAG(Login_date) OVER(PARTITION BY Emp_id ORDER BY login_date) AS lag_date,
        LEAD(Login_date) OVER(PARTITION BY Emp_id ORDER BY login_date) AS lead_date
    FROM Employees)
WHERE lag_date IS NOT NULL AND lead_date IS NOT NULL)
WHERE first = 1.0 AND second = 1.0
""")

cursor.execute("""
SELECT *,
        LAG(Login_date) OVER(PARTITION BY Emp_id ORDER BY login_date) AS lag_date,
        LEAD(Login_date) OVER(PARTITION BY Emp_id ORDER BY login_date) AS lead_date
    FROM Employees
""")
result = cursor.fetchall()
for r in result:
    print(r)




Connection established and database created successfully
Table created successfully
Data already exists


OperationalError: no such column: Login_date

In [ ]:
#Duplicate customer emailid
cursor.execute("""
CREATE TABLE IF NOT EXISTS Customer(
    customer_id INT PRIMARY KEY,
    email_id VARCHAR(50),
    name VARCHAR(50),
    joining_date DATE)
""")

conn.commit()
print("Table created successfully")

customer_data = [
    (101, 'alice@gmail.com', 'Alice', '2026-08-01'),
    (102, 'bob@gmail.com', 'Bob', '2026-08-02'),
    (103, 'alice@gmail.com', 'Alice New', '2026-08-05'),
    (104, 'john@gmail.com', 'John', '2026-07-30'),
    (105, 'bob@gmail.com', 'Bob New', '2026-08-04'),
    (106, 'mary@gmail.com', 'Mary', '2026-08-03')
]

cursor.execute("SELECT COUNT(*) FROM Customer")
count = cursor.fetchone()[0]

if count == 0:
    cursor.executemany("""
        INSERT INTO Customer (customer_id, email_id, name, joining_date)
        VALUES (?,?,?,?)""", customer_data)
    conn.commit()
    print("Data entered successfully")
else:
    print("Data already exists")
    

cursor.execute("""
DELETE FROM Customer
WHERE rowid IN (
SELECT rowid
FROM(
    SELECT rowid,
        RANK() OVER(PARTITION BY LOWER(email_id) ORDER BY joining_date DESC) AS rn
    FROM Customer)
WHERE rn > 1)
""")

cursor.execute("""
SELECT *
FROM Customer""")
result = cursor.fetchall()
for r in result:
    print(r)


In [ ]:
#7 Days rolling average sales
cursor.execute("""
CREATE TABLE IF NOT EXISTS daily_sales(
    sale_date TEXT,
    sales_amount INT)
""")

conn.commit()
print("Table created successfully")

sales_data = [
    ('2026-08-01', 1200),
    ('2026-08-02', 1400),
    ('2026-08-03', 1600),
    ('2026-08-04', 1800),
    ('2026-08-05', 1700),
    ('2026-08-06', 2000),
    ('2026-08-07', 2200),
    ('2026-08-08', 2100),
    ('2026-08-09', 2400),
    ('2026-08-10', 2300)
]

cursor.execute("SELECT COUNT(*) FROM daily_sales")
count = cursor.fetchone()[0]

if count == 0:
    cursor.executemany("""
        INSERT INTO daily_sales (sale_date, sales_amount)
        VALUES (?,?)""",sales_data)
    conn.commit()
    print("Data entered successfully")
else:
    print("Data already exists")


cursor.execute("""
SELECT *,
        ROUND(AVG(sales_amount) OVER (ORDER BY sale_date ROWS BETWEEN 7 PRECEDING AND CURRENT ROW),2)
FROM daily_sales
""")

result = cursor.fetchall()
for r in result:
    print(r)


In [ ]:
#placed orders with in seven days
cursor.execute("DROP TABLE IF EXISTS Orders")
conn.commit()

cursor.execute("""
CREATE TABLE Orders(
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date DATE
)
""")

conn.commit()
print("Table created successfully")

order_data = [
    (101, 1, '2026-08-01'),
    (102, 1, '2026-08-05'),
    (103, 2, '2026-08-02'),
    (104, 2, '2026-08-15'),
    (105, 3, '2026-08-03'),
    (106, 3, '2026-08-08'),
    (107, 4, '2026-08-04'),
    (108, 4, '2026-08-20')
]

cursor.execute("SELECT COUNT(*) FROM Orders")
count = cursor.fetchone()[0]

if count == 0:
    cursor.executemany("""
        INSERT INTO Orders (order_id, customer_id, order_date)
        VALUES (?,?,?)""", order_data)
    conn.commit()
    print("Data entered successfully")
else:
    print("Data already exists")


cursor.execute("""
WITH Rank_date AS(
SELECT *,ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY order_date) AS rn
FROM Orders)

SELECT *
FROM(
SELECT *, LEAD(order_date) OVER(PARTITION BY customer_id ORDER BY order_date) AS next_day
FROM(
SELECT *
FROM Rank_date
WHERE rn = 1 OR rn = 2))
WHERE (julianday(next_day) - julianday(order_date)) <= 7
""")

result = cursor.fetchall()
for r in result:
    print(r)

In [ ]:
#customer who bought every product under the category electronics
cursor.execute("DROP TABLE IF EXISTS Customers")
conn.commit()

cursor.execute("""
CREATE TABLE Customers(
    customer_id INTEGER PRIMARY KEY,
    name TEXT)
""")

cursor.execute("DROP TABLE IF EXISTS Orders")
conn.commit()

cursor.execute("""
CREATE TABLE Orders(
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    
    FOREIGN KEY(customer_id)
        REFERENCES Customers(customer_id))
""")

cursor.execute("DROP TABLE IF EXISTS products")
conn.commit()

cursor.execute("""
CREATE TABLE products(
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT
)
""")

cursor.execute("DROP TABLE IF EXISTS order_items")
conn.commit()

cursor.execute("""
CREATE TABLE order_items(
    order_id INTEGER,
    product_id INTEGER,
    
    FOREIGN KEY(order_id) 
        REFERENCES orders(order_id),
    FOREIGN KEY(product_id) 
        REFERENCES products(product_id)
)
""")

customers = [
    (1, 'Alice'),
    (2, 'Bob'),
    (3, 'Charlie')
]

orders = [
    (101, 1),
    (102, 1),
    (103, 1),
    (104, 2),
    (105, 2),
    (106, 3)
]

products = [
    (1, 'Laptop', 'Electronics'),
    (2, 'Mobile', 'Electronics'),
    (3, 'Headphones', 'Electronics'),
    (4, 'Chair', 'Furniture')
]

order_items = [
    (101, 1),
    (102, 2),
    (103, 3),
    (104, 1),
    (105, 2),
    (106, 4)
]

cursor.executemany(
    "INSERT INTO customers VALUES (?, ?)",
    customers
)

cursor.executemany(
    "INSERT INTO orders VALUES (?, ?)",
    orders
)

cursor.executemany(
    "INSERT INTO products VALUES (?, ?, ?)",
    products
)

cursor.executemany(
    "INSERT INTO order_items VALUES (?, ?)",
    order_items
)

conn.commit()
print("Table created and datas entered successfully")


cursor.execute("""
SELECT COUNT(*)
FROM products
WHERE category = 'Electronics'
""")

electronics_count = cursor.fetchone()[0]
print(electronics_count)

cursor.execute("""
WITH list AS (
SELECT o.order_id AS order_id,o.customer_id AS customer_id,p.product_id AS product_id,
       p.product_name AS product_name,p.category AS category
FROM orders AS o
INNER JOIN order_items AS i 
ON o.Order_id = i.Order_id
INNER JOIN products AS p
ON p.product_id = i.product_id)

SELECT l.category,c.customer_id,c.name
FROM list AS l
LEFT JOIN customers AS c
ON l.customer_id = c.customer_id
WHERE category = 'Electronics'
GROUP BY l.customer_id,c.name
HAVING COUNT(DISTINCT product_id) >= (SELECT COUNT(*) FROM products WHERE category = 'Electronics')
""")

result = cursor.fetchall()
for r in result:
    print(r)
    


In [ ]:
#customers with zero orders
cursor.execute(" DROP TABLE IF EXISTS customers")
conn.commit()

cursor.execute("""
CREATE TABLE customers (
customer_id INTEGER PRIMARY KEY,
name TEXT,
signup_date DATE
)
""")
conn.commit()

cursor.execute(" DROP TABLE IF EXISTS orders")
conn.commit()
cursor.execute("""
CREATE TABLE orders (
order_id INTEGER PRIMARY KEY,
customer_id INTEGER,
order_date DATE,
amount REAL,
FOREIGN KEY(customer_id) REFERENCES customers(customer_id)
)
""")
conn.commit()

customers = [
    (101, 'Alice', '2024-01-10'),
    (102, 'Bob', '2024-01-15'),
    (103, 'Charlie', '2024-02-01'),
    (104, 'David', '2024-02-18'),
    (105, 'Emma', '2024-03-10'),
    (106, 'Frank', '2024-03-20')
]

cursor.executemany("""
INSERT INTO customers( customer_id,name,signup_date)
VALUES (?,?,?)""",customers)
conn.commit()

orders = [
    (1001, 101, '2024-04-01', 1200),
    (1002, 101, '2024-04-15', 900),
    (1003, 102, '2024-04-18', 1500),
    (1004, 103, '2024-05-02', 2000),
    (1005, 105, '2024-05-20', 1750)
]

cursor.executemany("""
INSERT INTO orders(order_id,customer_id,order_date,amount)
VALUES(?,?,?,?)""",orders)
conn.commit()

cursor.execute("""
SELECT c.customer_id, c.name
FROM customers AS c
LEFT JOIN orders AS o
ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL
""")
result = cursor.fetchall()
for r in result:
    print(r)


In [ ]:
#Orders shipped late
cursor.execute("DROP TABLE IF EXISTS orders")
conn.commit()

cursor.execute("""
CREATE TABLE orders(
    order_id INTEGER PRIMARY KEY,
    order_date TEXT
)
""")

cursor.execute("DROP TABLE IF EXISTS shipments")
conn.commit()
cursor.execute("""
CREATE TABLE shipments(
    order_id INTEGER,
    shipped_date TEXT,
    status TEXT,
    
    FOREIGN KEY(order_id) 
        REFERENCES orders(order_id)
)
""")

orders = [
    (101, '2026-08-01'),
    (102, '2026-08-02'),
    (103, '2026-08-03'),
    (104, '2026-08-04'),
    (105, '2026-08-05')
]

shipments = [
    (101, '2026-08-02', 'Delivered'),
    (102, '2026-08-06', 'Delivered'),
    (103, '2026-08-04', 'Delivered'),
    (104, '2026-08-08', 'Delivered'),
    (105, '2026-08-06', 'Delivered')
]

cursor.executemany("""
     INSERT INTO orders(order_id,order_date)
     VALUES (?, ?)""",orders)

cursor.executemany("""
     INSERT INTO shipments(order_id,shipped_date,status)
     VALUES (?, ?, ?)""",shipments)

conn.commit()

print("Tables created and data inserted successfully!")

cursor.execute("""
SELECT *
FROM orders AS o
LEFT JOIN shipments AS s
ON o.order_id = s.order_id
WHERE (julianday(s.shipped_date) - julianday(o.order_date)) > 1
""")
result = cursor.fetchall()
for r in result:
    print(r)

In [ ]:
#Second highest salary
cursor.execute("DROP TABLE IF EXISTS departments")
conn.commit()
cursor.execute("""
CREATE TABLE departments (
    dept_id INTEGER PRIMARY KEY,
    dept_name TEXT
)
""")

cursor.execute("DROP TABLE IF EXISTS employees")
conn.commit()
cursor.execute("""
CREATE TABLE employees (
    emp_id INTEGER PRIMARY KEY,
    name TEXT,
    dept_id INTEGER,
    salary INTEGER,

    FOREIGN KEY (dept_id)
        REFERENCES departments(dept_id)
)
""")

departments = [
    (1, 'HR'),
    (2, 'IT'),
    (3, 'Finance')
]

cursor.executemany("""
    INSERT INTO departments(dept_id,dept_name)
    VALUES (?, ?)""",departments)

employees = [
    (101, 'Alice', 1, 60000),
    (102, 'Bob', 1, 50000),
    (103, 'Charlie', 1, 50000),
    (104, 'David', 1, 45000),
    (201, 'Emma', 2, 90000),
    (202, 'Frank', 2, 85000),
    (203, 'Grace', 2, 85000),
    (204, 'Henry', 2, 70000),
    (301, 'Irene', 3, 75000),
    (302, 'Jack', 3, 70000),
    (303, 'Kevin', 3, 65000)
]

cursor.executemany("""
    INSERT INTO employees(emp_id,name,dept_id,salary)
    VALUES (?, ?, ?, ?)""",employees)

conn.commit()

print("Tables created and data inserted successfully")

cursor.execute("""
SELECT *
FROM(
SELECT *,
       RANK() OVER(PARTITION BY d.dept_id ORDER BY e.salary DESC) AS rn
FROM employees AS e
LEFT JOIN departments AS d
ON e.dept_id = d.dept_id)
WHERE rn = 2
""")
result = cursor.fetchall()
for r in result:
    print(r)

In [ ]:
#Running total per customer
cursor.execute("DROP TABLE IF EXISTS payments")
conn.commit()

cursor.execute("""
CREATE TABLE payments (
    payment_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    payment_date TEXT,
    amount REAL
)
""")

payments = [
    (1, 101, '2026-08-01', 1000),
    (2, 101, '2026-08-05', 1500),
    (3, 101, '2026-08-10', 800),
    (4, 102, '2026-08-02', 2000),
    (5, 102, '2026-08-08', 1200),
    (6, 103, '2026-08-03', 900),
    (7, 103, '2026-08-09', 600)
]

cursor.executemany("""
INSERT INTO payments(payment_id, customer_id, payment_date, amount)
VALUES (?, ?, ?, ?)
""", payments)

conn.commit()

print("Payments table created and data inserted successfully")

cursor.execute("""
SELECT *,
       SUM(amount) OVER(PARTITION BY customer_id ORDER BY payment_date) AS cum_sum
FROM payments
""")
result = cursor.fetchall()
for r in result:
    print(r)

